# DeepSeek V4 Flash：稳定前缀与改前缀的 TTFT 对比

这个 Notebook 依次发送三次独立的流式请求：首次 `policy-v1`、相同 `policy-v1`、只将系统提示词首行改为 `policy-v2`。

观察指标是首个非空文本增量到达前的时间（TTFT）。TTFT 也受网络和服务负载影响，因此它是运行时观察，不单独证明缓存命中。

In [ ]:
# 代码块 1：准备运行环境并读取本次实验的密钥。密钥只留在 Notebook 内核内存。
import getpass
import json
import secrets
import statistics
import time
import urllib.error
import urllib.request

BASE_URL = 'https://api.deepseek.com/chat/completions'
MODEL = 'deepseek-v4-flash'
USER_PROMPT = '只回复：收到。'
WARMUP_REQUESTS = 2
SAMPLE_REQUESTS = 5
CACHE_BUILD_WAIT_SECONDS = 3
# 约 9.6 万输入 token：放大预填充差异；运行时间与未命中成本也会更高。
REFERENCE_REPEATS = 6_000
API_KEY = getpass.getpass('DeepSeek API Key: ').strip()
assert API_KEY, '未输入 API Key'
print(f'模型：{MODEL}；密钥仅保留在本次内核内存中。')

In [ ]:
# 代码块 2：构造长且稳定的系统提示词；marker 是唯一允许变化的首行。
def build_system_prompt(marker):
    stable_rule = '你是一个演示助手。请始终使用简洁中文回答，遵循给定任务，不要解释这段系统资料本身。'
    reference = '参考资料：稳定前缀应逐字保持一致；变化内容应放在请求末尾。'
    # marker 是唯一变量；其余内容始终逐字相同。
    return f'{marker}\n{stable_rule}\n' + (reference + '\n') * REFERENCE_REPEATS

v1 = build_system_prompt('policy-v1')
v2 = build_system_prompt('policy-v2')
assert v1.replace('policy-v1', 'policy-v2', 1) == v2
print(f'固定系统提示词长度：{len(v1):,} 字符；唯一改动：首行 v1 -> v2')

In [ ]:
# 代码块 3：定义流式请求与计时函数；这里只是准备函数，不会发请求或输出结果。
def stream_ttft(system_prompt):
    payload = {
        'model': MODEL,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': USER_PROMPT},
        ],
        'thinking': {'type': 'disabled'},
        'max_tokens': 16,
        'stream': True,
        'stream_options': {'include_usage': True},
    }
    request = urllib.request.Request(
        BASE_URL,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'},
        method='POST',
    )
    started_at = time.perf_counter()
    ttft_ms = None
    cache_usage = None
    with urllib.request.urlopen(request, timeout=120) as response:
        for raw_line in response:
            line = raw_line.decode('utf-8').strip()
            if not line.startswith('data:'):
                continue
            data = line[5:].strip()
            if data == '[DONE]':
                return ttft_ms, cache_usage
            event = json.loads(data)
            choices = event.get('choices', [])
            content = choices[0].get('delta', {}).get('content', '') if choices else ''
            if content and ttft_ms is None:
                ttft_ms = round((time.perf_counter() - started_at) * 1000, 1)
            usage = event.get('usage')
            if usage and 'prompt_cache_hit_tokens' in usage:
                cache_usage = (usage['prompt_tokens'], usage['prompt_cache_hit_tokens'], usage['prompt_cache_miss_tokens'])


In [ ]:
# 代码块 4：真正发起请求。先预热缓存，再比较稳定前缀组与每次都改首前缀的未命中组。
def run(label, marker):
    try:
        ttft_ms, usage = stream_ttft(build_system_prompt(marker))
        if usage is None:
            print(f'{label}: TTFT {ttft_ms:.1f} ms；未收到缓存 usage')
        else:
            prompt, hit, miss = usage
            print(f'{label}: TTFT {ttft_ms:.1f} ms；缓存命中 {hit}/{prompt} tokens，未命中 {miss}')
        return ttft_ms, usage
    except urllib.error.HTTPError as error:
        print(f'{label}: HTTP {error.code}（请检查 API Key、余额与模型权限）')
    except urllib.error.URLError as error:
        print(f'{label}: 网络请求失败：{error.reason}')

print('实验开始：将依次发送 12 次请求，通常需要十几秒。', flush=True)
for index in range(WARMUP_REQUESTS):
    run(f'预热 #{index + 1}', 'policy-v1')
time.sleep(CACHE_BUILD_WAIT_SECONDS)

def run_group(label, markers):
    results = [run(f'{label} #{index}', marker) for index, marker in enumerate(markers, 1)]
    ttfts = [ttft for ttft, _ in results if ttft is not None]
    usages = [usage for _, usage in results if usage is not None]
    print(f'{label} 中位 TTFT：{statistics.median(ttfts):.1f} ms')
    if usages:
        prompt = sum(usage[0] for usage in usages)
        hit = sum(usage[1] for usage in usages)
        print(f'{label} 缓存命中率：{hit / prompt:.1%} ({hit}/{prompt} tokens)')

run_group('稳定前缀组', ['policy-v1'] * SAMPLE_REQUESTS)
# 每次运行、每个样本都从第一个 token 开始使用新的随机值，避免复用上一次 Notebook 运行写入的缓存。
miss_markers = [f'{secrets.token_hex(12)}-cache-miss' for _ in range(SAMPLE_REQUESTS)]
run_group('改首前缀组', miss_markers)
print('以 usage 中的缓存 token 判定命中；TTFT 仅作辅助观察。')